# Week 2 · Class 2 — NumPy essentials & pandas fundamentals 🐼

**Plan for today (120 min)**

| Part | Topic | Time |
|---|---|---|
| 1 | NumPy essentials: arrays, shapes, operations, masks | 20 min |
| 2 | pandas fundamentals: Series, DataFrames, selecting, cleaning, grouping, combining | 90 min |
| 3 | Lambda functions & `.apply`: answering questions with custom rules | 10 min |

**Goal:** you won't learn *every* method today. You'll learn the **core vocabulary** of NumPy and pandas, so that you can
read the documentation, search for what you need, and solve Saturday's labs on your own.

After this class you should be able to:
- create NumPy arrays, check their `shape`, reshape/transpose them, compute statistics and label values with conditions
- create pandas `Series` and `DataFrame`s from lists and dictionaries, and read a CSV file
- select rows and columns (`[]`, `.loc`, `.iloc`, boolean filters) and compute statistics
- detect missing values, fix column types, set an index
- summarise data with `groupby`, and combine tables with `concat`, `merge` and `join`
- write `lambda` functions and use `.apply` to create new columns from your own rules

> 🧑‍💻 **Live-coding version** — we write the code together in class.

## 🧭 How to find what you don't know

Nobody remembers every method. Professionals use these tricks all the time:

- `np.zeros?` or `help(np.zeros)` → shows the documentation of a function inside the notebook
- Type `df.` and press **Tab** → lists every method available on your object
- Official docs: [NumPy reference](https://numpy.org/doc/stable/reference/index.html) · [pandas API reference](https://pandas.pydata.org/docs/reference/index.html)
- Searching "pandas how to ..." works very well — the answer is usually one method name away

---
# Part 1 — NumPy essentials (20 min) 🔢

**NumPy** is the library for working with numbers in Python. Its central object is the **array**: a grid of values that all have the **same type**.

Why not just Python lists?
- arrays are **much faster** for maths on many numbers
- you can do maths on the whole array at once (no loops needed)
- pandas is built on top of NumPy: every pandas column is a NumPy array underneath

In [ ]:
# Import NumPy with its standard nickname "np"
import numpy as np

# Which version do we have?
print(np.__version__)

## 1.1 Creating arrays and inspecting them

The 4 attributes you'll check all the time:

| Attribute | Meaning |
|---|---|
| `.shape` | size along each dimension, e.g. `(2, 3)` = 2 rows, 3 columns |
| `.ndim` | number of dimensions (1 = vector, 2 = matrix, 3+ = tensor) |
| `.size` | total number of elements |
| `.dtype` | type of the elements (`int64`, `float64`, ...) |

In [ ]:
# A 1D array (a vector) from a list
# A 2D array (a matrix) from a list of lists: each inner list is a row

### Ready-made arrays

Very often you don't type values by hand: you ask NumPy to build an array of a given **shape**.
The shape is always passed as a **tuple**, e.g. `(2, 3, 4)`.

In [ ]:
# Arrays full of 0s or 1s
# An "empty" array: memory is reserved but values are garbage until you fill them
# Regularly spaced values

### Random arrays

Random data is great for practising and for simulations. The modern way is to create a **random generator** with a seed
(the seed makes the "random" numbers reproducible: same seed → same numbers).

In [ ]:
# Create a random generator (seed = 42 makes results reproducible)
# Uniform floats between 0 and 1, with a given shape
# Random integers between 0 and 10 (10 excluded)
# Normal distribution with mean 5 and standard deviation 2
# You'll also see the older style in tutorials: it still works

## 1.2 Indexing and slicing

Indexing starts at **0**. For a 2D array you write `array[row, column]`.
`:` means "everything", and `start:stop` takes a range (stop excluded).

In [ ]:
# One element: row 1, column 3
# A whole column
# Rows 1 and 2, all columns
# Last two columns (negative indices count from the end)

## 1.3 Element-wise operations and broadcasting

Operations like `+`, `-`, `*`, `/` work **element by element**: no loop needed.

⚠️ To combine two arrays, their shapes must be **compatible**. Same shape always works.
A single number (a *scalar*) also works: NumPy "broadcasts" it to every element.

In [ ]:
# What happens with incompatible shapes?
# try/except lets us see the error without stopping the notebook

## 1.4 Reshape and transpose

- `.reshape(new_shape)` re-arranges the **same elements** into a new shape. The total size must stay the same. Use `-1` to let NumPy compute one dimension.
- `.T` swaps rows and columns of a matrix. For a 3D array it **reverses** all axes.
- `.transpose(axes)` lets you choose the **new order of the axes** yourself.

In [ ]:
# Transposing a 3D array: choose where each old axis goes
# We want shape (2, 3, 5): new axis order = old axes (1, 2, 0)

## 1.5 Statistics on arrays

`.sum()`, `.mean()`, `.min()`, `.max()`, `.std()` work on the whole array, or along one **axis**:
- `axis=0` → collapse the rows (one result per **column**)
- `axis=1` → collapse the columns (one result per **row**)

## 1.6 Masks and conditional labels

A comparison like `scores > 12` gives an array of `True`/`False` called a **mask**. You can use it to:
- **select** values: `scores[mask]`
- **count** them: `mask.sum()` (True counts as 1)
- **label** values: `np.where` (one condition) or `np.select` (several conditions)

These are the tools you need to label values in an array without writing loops.

In [ ]:
# np.where(condition, value_if_true, value_if_false)

In [ ]:
# np.select: list of conditions + list of matching values (first True wins)
# Same idea with numbers, and same shape as the input

### 🏋️ NumPy practice (3 min)

1. Create a random array `d` of shape `(2, 3, 4)` with values between 1 and 2 (hint: `rng.random` gives values in [0, 1)).
2. Store its min, max and mean in `d_min`, `d_max`, `d_mean`.
3. Create `f`, same shape as `d`, containing `0` where `d` equals its minimum, `100` where it equals its maximum, `25` below the mean and `75` above the mean.

In [ ]:
# your code here


---
# Part 2 — pandas fundamentals (90 min) 🐼

**pandas** is the library for **tables** (like Excel or SQL tables) in Python. Two objects to know:

| Object | What it is | Analogy |
|---|---|---|
| `Series` | one column of values with an **index** (labels) | one Excel column |
| `DataFrame` | several Series side by side, sharing the same index | a whole Excel sheet |

In [ ]:
import pandas as pd
print(pd.__version__)

## 2.1 Series (8 min)

A Series is a 1D NumPy array **plus labels** (the index). By default the index is `0, 1, 2, ...`.

In [ ]:
# Third value: position 2 (counting starts at 0)

In [ ]:
# A Series can have custom labels
# Statistics work directly

💡 **Rule of thumb:** `.loc` uses labels, `.iloc` uses integer positions. When in doubt, be explicit and use one of them.

## 2.2 Creating DataFrames (7 min)

The two most common ways to build a DataFrame by hand:
- from a **dictionary**: keys become column names, lists become columns
- from a **list of lists**: each inner list becomes a **row**

In [ ]:
# From a dictionary of columns

In [ ]:
# From a list of lists: each sublist is a row
# Rename ALL columns at once by assigning a list
# Or rename only some columns with a dictionary {old: new}

In [ ]:
# The key attributes of a DataFrame

## 2.3 Reading a file and first look (10 min)

We'll use a real dataset: orders from a restaurant chain (Chipotle). Each row is one item in an order.

`pd.read_csv` reads CSV files. This file is separated by **tabs**, so we pass `sep="\t"`.
It also works with a URL or a local path like `"data/chipotle.tsv"`.

In [ ]:
url = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/chipotle.tsv"
orders = pd.read_csv(url, sep="\t")

In [ ]:
# First 5 rows (use .tail() for the last ones)

In [ ]:
# How big is it?
# Column names, types and number of non-missing values

In [ ]:
# Summary statistics of numeric columns

In [ ]:
# For text columns: how many different values, and the most frequent ones

## 2.4 Selecting data (15 min)

| You want | Syntax |
|---|---|
| one column (→ Series) | `df["col"]` |
| several columns (→ DataFrame) | `df[["col1", "col2"]]` ← note the **double brackets** |
| rows/columns by label | `df.loc[rows, cols]` |
| rows/columns by position | `df.iloc[rows, cols]` |
| rows matching a condition | `df[condition]` |

In [ ]:
# One column → Series

In [ ]:
# Several columns → DataFrame (double brackets: a list inside [])

In [ ]:
# .iloc: by position → first 3 rows, first 2 columns
# .loc: by label → rows with index 1 to 3 (INCLUDED with .loc!), columns age and job

### Boolean filtering

Exactly like NumPy masks: a condition gives a Series of `True`/`False`, and `df[condition]` keeps the `True` rows.

To combine conditions use `&` (and), `|` (or), `~` (not) — and **always put each condition in parentheses**.

In [ ]:
# People older than 35

In [ ]:
# Web analysts AND older than 28

In [ ]:
# Several possible values: .isin()

In [ ]:
# Counting rows that match: len() of the filtered table, or .sum() of the mask
# A statistic on the filtered rows only

## 2.5 Cleaning basics (12 min)

Real data is messy. Before analysing, always check:
1. **missing values**
2. **column types** (numbers stored as text?)
3. **duplicates / identifiers**

In [ ]:
# 1. Missing values per column

In [ ]:
# Two common options: drop the rows, or fill the gaps

In [ ]:
# 2. item_price is text like "$2.39 " → we can't do maths with it
# Remove "$", remove spaces, convert to float

In [ ]:
# 3. Setting a column as the index
# drop=False keeps the column in the table as well

In [ ]:
# Is a column (or a combination of columns) a unique identifier?
# For several columns: look for duplicated combinations

## 2.6 New columns and statistics (6 min)

In [ ]:
# Arithmetic between columns is element-wise (like NumPy)
# A custom rule on each value: .apply with a lambda function

In [ ]:
# Column statistics: sum, mean, median, min, max
# Sorting

## 2.7 GroupBy: split → apply → combine (12 min)

`df.groupby("col")` splits the table into groups with the same value of `col`, applies a calculation to each group and combines the results.

In [ ]:
# Average age per job

In [ ]:
# Several statistics at once with .agg

In [ ]:
# Real data: top 5 items by revenue

In [ ]:
# Revenue per order, then the average order value

## 2.8 Combining tables (12 min)

| Function | What it does | SQL analogy |
|---|---|---|
| `pd.concat([df1, df2])` | stacks tables **on top of each other** (or side by side with `axis=1`) | `UNION` |
| `pd.merge(left, right, on="key", how=...)` | matches rows using a **key column** | `JOIN` |
| `left.join(right)` | like merge, but matches on the **index** | `JOIN` on index |

In [ ]:
sales_1 = pd.DataFrame({"sales_id": ["001", "002", "003", "004"],
                        "people_id": [1, 4, 2, 1],
                        "product_id": ["X789", "X999", "X789", "X990"]})
sales_2 = pd.DataFrame({"sales_id": ["005", "006", "007"],
                        "people_id": [0, 3, 2],
                        "product_id": ["X789", "X999", "X789"]})
products = pd.DataFrame({"product_id": ["X789", "X999", "X990"],
                         "product_desc": ["Apple", "Banana", "Orange"]})
customers = pd.DataFrame({"id": [0, 1, 2, 3, 5],
                          "name": ["Paul", "Perrine", "Moussa", "Michel", "Anne"],
                          "age": [None, 67, 24, 76, 47]})

In [ ]:
# Stack rows: same columns, one table under the other
# The index is repeated (0..3 then 0..2) → ignore_index=True renumbers it

In [ ]:
# axis=1 puts tables SIDE BY SIDE, aligning on the index

In [ ]:
# Merge on a common column name

### What if keys don't match?

`people_id` 4 has no customer, and customer 5 bought nothing. The `how` argument decides what to keep:

| `how=` | keeps |
|---|---|
| `"inner"` (default) | only keys present in **both** tables |
| `"left"` | all rows of the left table |
| `"right"` | all rows of the right table |
| `"outer"` | all rows of both, with `NaN` where data is missing |

When the key columns have **different names**, use `left_on=` and `right_on=`.

In [ ]:
# .join matches on the INDEX → set the key as index first

## 2.9 A taste of dates and text (5 min)

pandas has two "toolboxes" for special columns:
- `.dt` for dates and times (after converting with `pd.to_datetime`)
- `.str` for text (we already used `.str.replace` and `.str.strip`)

In [ ]:
# Text columns: .str methods apply to every value

In [ ]:
# Dates: text → datetime, then extract parts with .dt

---
# Part 3 — Lambda functions & `.apply` (10 min) λ

Sometimes pandas has no ready-made method for the rule you need. Then you write the rule as a **function** and ask pandas to
**apply** it to every value of a column.

### What is a lambda?

A `lambda` is a **tiny function without a name**, written in one line. These two are equivalent:

```python
def add_vat(price):
    return price * 1.19

add_vat = lambda price: price * 1.19
```

Read it as: *"take `price`, give back `price * 1.19`"*. Use `lambda` for short one-line rules, and `def` when the rule needs several lines.

In [ ]:
# A lambda is a normal function: you can store it and call it
# Lambdas can take several inputs

### `.apply` on a column

`df["col"].apply(function)` calls the function **once per value** and returns a new column with the results.

In [ ]:
# Question 1: how are orders split into price bands?

In [ ]:
# Question 2: which protein is the most popular in burritos, bowls and tacos?

### When the rule is longer: use `def`

`.apply` accepts **any** function, not only lambdas. If your rule needs several `if`s, a named function is easier to read.

In [ ]:
# Question 3: what share of the revenue comes from each category?

In [ ]:
# Question 4: do people pay more when they add guacamole?

### `.apply` with `groupby`: one rule per group

After a `groupby`, `.apply(lambda group: ...)` receives **all the values of one group** at once.

In [ ]:
# Question 5: what share of orders includes at least one drink?

💡 **Good habit:** `.apply` runs Python code row by row, so it is slower on big data. If a vectorised version exists
(arithmetic between columns, `.str.contains`, `np.where`...), prefer it. Use `.apply` when you need **custom logic**.

### 🏋️ Your turn (in class)

**A.** Create a column `n_toppings` with the number of choices in `choice_description` (hint: count the commas and add 1; `"No choice"` should give 0). What is the average `unit_price` for each number of toppings?

In [ ]:
# your code here


**B.** Using `order_totals` (revenue per order, from section 2.7), label each order `"small"` (< 15), `"medium"` (15–30) or `"large"` (> 30) with `.apply`. How many orders are in each group?

In [ ]:
# your code here


---
## 🏋️ Practice (at home or if time allows)

Use the `orders` table (already cleaned above).

**1.** How many rows have a `quantity` greater than 1 **and** an `item_price` above 10?

In [ ]:
# your code here


**2.** What is the average `item_price` of items whose name contains `"Bowl"`?

In [ ]:
# your code here


**3.** For each `item_name`, compute the total `quantity` sold. Which 3 items sold the most units?

In [ ]:
# your code here


**4.** Using `sales`, `products` and `customers`: build a table with one row per sale, the product description and the customer name. Keep all sales, even those without a known customer. How many sales have no customer name?

In [ ]:
# your code here


---
## 📋 Cheat sheet

**NumPy**
- create: `np.array`, `np.zeros`, `np.ones`, `np.empty`, `np.arange`, `np.linspace`, `rng = np.random.default_rng(seed)` → `rng.random`, `rng.integers`, `rng.normal`
- inspect: `.shape`, `.ndim`, `.size`, `.dtype`
- reshape: `.reshape`, `.T`, `.transpose(axes)`
- stats: `.sum()`, `.mean()`, `.min()`, `.max()`, `.std()` (with `axis=`)
- conditions: masks `a[a > x]`, `np.where`, `np.select`

**pandas**
- create/read: `pd.Series`, `pd.DataFrame`, `pd.read_csv`
- inspect: `.head()`, `.tail()`, `.shape`, `.info()`, `.describe()`, `.dtypes`, `.value_counts()`, `.nunique()`
- select: `df["col"]`, `df[["a", "b"]]`, `.loc`, `.iloc`, `df[cond]` with `&`, `|`, `~`, `.isin`
- clean: `.isna().sum()`, `.dropna()`, `.fillna()`, `.astype()`, `.rename()`, `.set_index()`, `.is_unique`, `.duplicated()`
- transform: column arithmetic, `.apply(lambda ...)` or `.apply(my_function)`, `groupby(...).apply(...)`, `.sort_values()`
- summarise: `.groupby().agg()`
- combine: `pd.concat`, `pd.merge(how=, on= / left_on= right_on=)`, `.join`
- special columns: `.str.*`, `pd.to_datetime`, `.dt.*`

**Next class:** we go deeper into exploratory data analysis and statistics (mean vs median, spread, outliers).